In [ ]:
import yfinance as yf
import pandas as pd
import os
from dotenv import load_dotenv
import databento as db
dat = yf.download(["^NDX", "^VXN"], start="2026-10-27", end="2026-11-27", auto_adjust=False, )

display(dat.index.tz)

dat.head()

In [ ]:
long = dat.stack(level="Ticker")
long = long.reset_index()
long.columns = long.columns.str.lower()
long = long.rename(columns={"ticker": "instrument"})
long = long[["date", "instrument", "open", "high", "low", "close", "adj close", "volume"]]
long.columns.name = None
long.head()

In [ ]:
display(long.duplicated(subset=["date", "instrument"], keep=False).sum())
len(long)

In [ ]:
load_dotenv()
db_api_key = os.getenv("DATABENTO_API_KEY")

client = db.Historical(db_api_key)

try:
    cost = client.metadata.get_cost(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    print(cost, 'USD')
except db.BentoClientError as e:
    print(f"Error: {e}")


In [ ]:
try:
    db_data = client.timeseries.get_range(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    db_data = db_data.to_df()
except db.BentoClientError as e:
    print(f"Error: {e}")

In [ ]:
db_data_long = db_data.reset_index()
db_data_long = db_data_long.rename(columns={"ts_event": "ts"})
db_data_long["instrument"] = "NQ"
db_data_long = db_data_long[["ts", "instrument", "instrument_id", "open", "high", "low", "close", "volume"]]
db_data_long.head()

In [ ]:
display(db_data_long.duplicated(subset=["ts", "instrument"], keep=False).sum())


- bars_1d (yfinance): columns date, instrument, open, high, low, close, adj close, volume; key (date, instrument)
- bars_1m (Databento): columns ts (UTC), instrument, instrument_id, open, high, low, close, volume; key (ts, instrument)
- Fact: both keys are unique on the test periods (0 duplicates), 44 and 9,660 rows as expected
- Impact: the key makes loading idempotent: reloading a day replaces its rows instead of duplicating them; a duplicate check runs before each load